# Position and Phase Transition Studies

In this notebook, we explore the fundamental **equations of motion** and compare the **numerical methods** used in our [MD simulation](). This is achieved by visualizing **particle configurations** in the system. Additionally, we employ the counting of **neighboring particles** throughout the simulation as an indicator of particle density and the current **phase of the system** (solid, liquid, or gas). 

As part of this study, we use the output data from the [MD code]() to analyze both **phase transitions** (using a thermostat to control temperature during annealing and heating processes) and other routine investigations.  
  
The visualization techniques employed include:
- **Snapshots** of particle positions at specific times.
- **Particle trajectories** over a specific time window, highlighting start and end positions.
- **Dynamic animations** to observe the effects of interactions on particle movements over time.
- **Neighbor count plots** over time, to identify critical points in phase transitions and determine the critical temperature during the annealing process.

---

### Outline:
1. **Many-Body Problem and Equations of Motion**: Overview of the many-body problem in classical mechanics and the application of Newton's laws of motion.
2. **Integration Methods**: A comparison of common numerical methods used to solve equations of motion in many-body systems:
   - **Euler Method**
   - **Symplectic Euler Method**
   - **Velocity Verlet Method**
3. **Phase Transition Studies**: Analysis of particle neighbors over time to investigate phase transitions.

---


## 1. Many-Body Problem and Equations of Motion

Molecular Dynamics (MD) simulations aim to track the precise **positions** and **velocities** of each particle in a system over time. This is a classical mechanics problem where, given the governing **forces** or the **potential energy** (the negative gradient of which gives the force), we can solve the equations of motion to determine the system's evolution at each time step. This includes key properties such as accelerations, velocities, positions, kinetic energy, temperature, and other physical observables.

### Degrees of Freedom and Equations of Motion
In a 2D system, each particle has two, or three degrees of freedom: **x-position**, **y-position**, and **orientation** $ \phi $. In our simulation, we use [`particleDot`]() for particles that can only have translational motion, and [`particleOriented`]() for the ones that can have orientation($\phi$).
Therefore, the motion must be described by the equations of motion for both translational and rotational dynamics.

Newton's second law describes the translational motion:

$$
\mathbf{F} = m \mathbf{a}, \quad \mathbf{a} = \frac{d\mathbf{v}}{dt}, \quad \mathbf{v} = \frac{d\mathbf{r}}{dt}
$$

where $ \mathbf{r} = (x, y) $ and $ \mathbf{F} = (F_x, F_y) $ are the position and force vectors in Cartesian coordinates.

For rotational motion, the angular acceleration is governed by the torque $ \tau $:

$$
\tau = I \alpha, \quad \alpha = \frac{d\omega}{dt}, \quad \omega = \frac{d\phi}{dt}
$$

where:
- $ \phi $ is the orientation angle,
- $ \omega $ is the angular velocity,
- $ \alpha $ is the angular acceleration,
- $ I $ is the moment of inertia.

### Force Components and Generalization
In our simulation, the force components can be divided into two parts:
- **Translational Force Components**: $ F_x $ and $ F_y $ are used to update the linear positions $x$ and $y$ and their respective velocities.
- **Rotational Force Component**: The tangential component of the force contributes to the **torque**, which is used to update the orientation $ \phi $ and angular velocity $ \omega $.

The force is derived from a potential function $ U(\mathbf{r}, \phi) $. The generalized force expressions for each degree of freedom are:

$$
F_x = -\frac{\partial U}{\partial x}, \quad
F_y = -\frac{\partial U}{\partial y}, \quad
\tau = -\frac{\partial U}{\partial \phi}
$$

### Generalized Approach in the Code
The simulation uses a **generalized format** for handling different types of particles:
- [`particleDot`](): Non-oriented particles, described only by translational degrees of freedom.
- [`particleOriented`](): Oriented particles with both translational and rotational dynamics.

Each particle type has its own potential and force calculations implemented in separate **acceleration functions**. However, the code employs a **generalized interface** using C++ templates and traits (template template) to ensure flexibility in setting and retrieving positions, velocities, and forces uniformly across different particle types.

This design allows for a unified treatment of both translational and rotational motions while maintaining the flexibility to choose different potentials and force calculations.

---



## 2. Numerical Integration Methods

Solving the equations of motion analytically becomes impractical for systems with more than few interacting particles. This complexity necessitates the use of computational and numerical methods, known as **numerical integration methods**, to approximate the solutions.

Several numerical integration methods are employed in MD simulations to solve the equations of motion:

1. **Euler Method**:
   - **Description**: A straightforward method based on the Taylor expansion, developed by Leonhard Euler.
   - **Formulation**:
     $$ x(t + \Delta t) = x(t) + v(t) \cdot \Delta t $$
     $$ v(t + \Delta t) = v(t) + a(t) \cdot \Delta t $$
   - **Pros and Cons**: Easy to implement but less accurate and not symplectic, which can lead to energy drift over time.

2. **Symplectic Euler Method** (also known as Semi-Implicit Euler or Euler-Cromer):
   - **Description**: An improved version of the Euler method that is symplectic, meaning it preserves the geometric properties of Hamiltonian systems, leading to better energy conservation.
   - **Formulation**:
     $$ v(t + \Delta t) = v(t) + a(t) \cdot \Delta t $$
     $$ x(t + \Delta t) = x(t) + v(t + \Delta t) \cdot \Delta t $$
   - **Pros and Cons**: More accurate than the standard Euler method and preserves symplectic structure, but still only first-order accurate.

3. **Velocity Verlet Method**:
   - **Description**: A widely used method in MD simulations that offers a good balance between accuracy and computational efficiency.
   - **Formulation**:
     $$ x(t + \Delta t) = x(t) + v(t) \cdot \Delta t + \frac{1}{2} \cdot a(t) \cdot (\Delta t)^2 $$
     $$ v(t + \Delta t) = v(t) + \frac{1}{2} \cdot [a(t) + a(t + \Delta t)] \cdot \Delta t $$
   - **Pros and Cons**: Second-order accurate, symplectic, and time-reversible, making it suitable for long-term simulations.

In computational methods, there's always a trade-off between accuracy and computational cost. While higher-order methods like Runge-Kutta offer greater accuracy, they come with increased computational expense. The **Velocity Verlet** method provides a favorable balance, making it well-suited for our MD simulations.

In our [MD simulation](), we've implemented the **Euler**, **Symplectic Euler**, and **Velocity Verlet** methods. Users can select the desired integration method as a runtime parameter to observe and compare the outcomes of each approach.

---




Below, you can find various plots displaying particle **trajectories**, **snapshot of system**, and **animations** of particle movements, providing a clearer view of the system's behavior over time. Additionally, plots of the system's **kinetic**, **potential**, and **total energies** are included for a more comprehensive analysis of the system's dynamics and phase behavior:  

1. **plot trajectories in a time interval** (both particleDot, and particleOriented):

In [1]:
import numpy as np
import matplotlib.pyplot as plt

def read_positions(filename, particle_type):
    type_size = 2 if particle_type == "dot" else 3
    data = np.loadtxt(filename)[:, 1:]
    num_steps, num_particles = data.shape[0], data.shape[1] // type_size
    return data.reshape(num_steps, num_particles, type_size)

def plot_trajectory(path, particle_type, t_min, t_max):
    positions = read_positions(path, particle_type)
    print("Size of position file = ", positions.shape)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4), dpi=150)

    for i in range(positions.shape[1]):
        ax[0].plot(positions[t_min:t_max, i, 0], positions[t_min:t_max, i, 1], 'o', ms=1, alpha=0.6)

    ax[0].set_title("Particle Trajectories")
    ax[0].set_xlabel("X Position")
    ax[0].set_ylabel("Y Position")
    ax[0].grid(linestyle='--', alpha=0.5)

    ax[1].scatter(positions[t_min, :, 0], positions[t_min, :, 1], marker='o', color='blue', label='Initial Position')
    ax[1].scatter(positions[t_max, :, 0], positions[t_max, :, 1], marker='x', color='red', label='Final Position')

    ax[1].set_title("Initial vs. Final Positions")
    ax[1].set_xlabel("X Position")
    ax[1].set_ylabel("Y Position")
    ax[1].legend()
    ax[1].grid(linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


In [ ]:
path = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan7_dotLong_COMang_diffSeed/N_particle_PosMD.dat"
plot_trajectory(path, "dot", 390000, 400000)

---  

2. **plot snapshot of system configurations in phase transition** (particleOriented):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

def plotSnapShot(directory, start, end):
    items = os.listdir(directory)
    configs = sorted(items)
    temperature = [name.split('_')[-1].replace('.dat', '') for name in configs]

    for i in range(start, end):
        file_path = os.path.join(directory, configs[i])
        res = np.loadtxt(file_path)
        print(res.shape)

        plt.figure(figsize=(6, 4))
        
        scatter = plt.scatter(res[:, 0], res[:, 1], c=res[:, 2], cmap='twilight', s=50, alpha=0.8, vmin=0, vmax=360)
        
        cbar = plt.colorbar(scatter)
        cbar.set_label('φ in degrees')
        
        plt.gca().set_aspect('equal', adjustable='box')
        plt.xlim(0, 20)
        plt.ylim(0, 20)
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.title(f'Positions + Orientation φ  (T={temperature[i]})')
        plt.grid(True)

        plt.show()

file1 = "/home/hadis/custom_vector/buildParticleOriented/buildVS/nov18/cooling_configs/"

plotSnapShot(file1, 0, 1)

---  

3. **Animation of particles movement** (both types):

In [17]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm


def map_to_frame(x, y, frame_size, offset=0):
    scale = frame_size / 20
    return int(x * scale + offset), int(y * scale + offset)

def phi_to_color(phi):
    norm_phi = phi / (2 * np.pi) 
    color = cm.twilight(norm_phi)
    bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
    return bgr_color

def temperature_to_color(temp):
    norm_temp = (temp - 0.1) / (1 - 0.1)
    color = cm.coolwarm(norm_temp)
    bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
    return bgr_color
    
def animation_particles(file, particle_type, skip_rows, output_name, phase_transition = False) :
    frame_size = 800
    particle_radius = 10

    type_size = 2 if particle_type == "dot" else 3

    positions = []
    with open(file, 'r') as file:
        for i, line in enumerate(file):
            if i % skip_rows == 0:
                data = line.strip().split()
                data = list(map(float, data[1:]))
                positions.append(data)
    positions = np.array(positions)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_name, fourcc, 10.0, (frame_size + 200, frame_size))

    high_temp = 1.0
    low_temp = 0.1
    num_frames = len(positions)
    temperatures = np.linspace(high_temp, low_temp, num_frames // 2).tolist() + np.linspace(low_temp, high_temp, num_frames - num_frames // 2).tolist()

    for i, frame_data in enumerate(positions):
        frame = np.ones((frame_size, frame_size + 200, 3), dtype=np.uint8) * 255

        # Draw bounding box around particles
        cv2.rectangle(frame, (0, 0), (frame_size, frame_size), (0, 0, 0), 2)

        # Draw particles
        for j in range(0, len(frame_data), type_size):
            x, y = frame_data[j], frame_data[j + 1]
            cx, cy = map_to_frame(x, y, frame_size)

            if type_size == 3:
                phi = frame_data[j + 2] + np.pi
                color = phi_to_color(phi)
            else:
                color = (120, 120, 120)

            cv2.circle(frame, (cx, cy), particle_radius, color, -1)
        if phase_transition == True :
            # Draw temperature bar with gradient
            bar_x_start = frame_size + 50
            bar_x_end = frame_size + 150
            bar_y_start = 50
            bar_y_end = frame_size - 50
            for y in range(bar_y_start, bar_y_end):
                t = (y - bar_y_start) / (bar_y_end - bar_y_start)
                color = cm.coolwarm(1 - t)
                bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
                cv2.line(frame, (bar_x_start, y), (bar_x_end, y), bgr_color, 1)

            # Draw indicator for current temperature
            temp = temperatures[i]
            indicator_y = int(bar_y_end - (temp - 0.1) / (1 - 0.1) * (bar_y_end - bar_y_start))
            cv2.arrowedLine(frame, (bar_x_end + 10, indicator_y), (bar_x_end + 50, indicator_y), (0, 0, 0), 2, tipLength=0.3)

        out.write(frame)

        if i % 50 == 0:
            print(f"Processing frame {i}/{len(positions)}")

    out.release()
    print("Video saved as", output_name)

    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("Particle Movement at Last Frame")
    plt.show()

In [ ]:
file_path = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan9_orSingleRandom/N_particle_PosMD.dat"
particle_type = "oriented"
skip_rows = 100

animation_particles(file_path, particle_type, skip_rows, "particle_animation.mp4", phase_transition=False)

---  
4. **Total Energy plots**:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_energy(file, label="", style="-", fig=None, ax=None):
    kineticEnergies = np.loadtxt(f"/home/hadis/custom_vector/buildParticleOriented/buildVS/{file}/N_particle_KEsAveMD.dat")
    potentialEnergies = np.loadtxt(f"/home/hadis/custom_vector/buildParticleOriented/buildVS/{file}/N_particle_PotentialEnergyMD.dat")

    KEx = kineticEnergies[:, 0]
    KEy = kineticEnergies[:, 1]
    KEphi = kineticEnergies[:, 2]
    POTE = np.sum(potentialEnergies, axis=1) / (potentialEnergies.shape[1] * 2)

    totalE = KEx + KEy + KEphi + POTE
    kineticE = KEx + KEy

    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(8, 4.5), dpi=100)
    
    ax.plot(kineticE, label=f"{label} Kinetic Energy", linestyle=style, linewidth=1.5)
    ax.plot(POTE, label=f"{label} Potential Energy", linestyle=style, linewidth=1.5)
    ax.plot(totalE, label=f"{label} Total Energy", linestyle=style, linewidth=1.5)
    
    ax.set_xlabel("Time Steps", fontsize=14)
    ax.set_ylabel("Energy", fontsize=14)
    ax.set_title("Energy Evolution Over Time", fontsize=16, fontweight="bold")
    ax.legend(fontsize=12, loc="center right")
    ax.grid(True, which="both", linestyle="--", linewidth=0.5)
    
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)
    
    print(f"First 10 Total Energy Values: {totalE[:10]}")
    
    return fig, ax

fig, ax = plot_energy("jan9_orSingleRandom", label="", style="-")
plt.show()


---

## 3. Phase Transition Studies

Studying phase transitions requires controlling the system temperature, often using thermostats, and performing simulations at varying temperatures. To achieve this, we run the [main MD code](/home/hadis/custom_vector/test/MDsimulation/nParticleMD.cpp) multiple times at different temperature settings, using the results from one run (particle positions and velocities) as the initial conditions for the next simulation. For efficiency, we automate this process using [bash scripts](/home/hadis/custom_vector/bin/MDsimulation/runMD.sh) to manage batch runs and data handling.

In phase transition studies, several properties of the system are monitored, including:
- **Particle configurations**: Changes in spatial arrangements of particles.
- **Kinetic and Potential Energy**: Variations during temperature changes.

### Neighbor Counting as a Phase Indicator
An important metric for phase analysis is the **number of neighbors** around each particle and the **average neighbor count** for the entire system. The following image illustrates the concept of neighbor counting and how it relates to particle interactions:
<p align="center">
    <img src="/home/hadis/custom_vector/recourses/neighborsCounters.png" alt="Image description" width="500"/>
</p>
By tracking the average number of neighbors for different cutoff distances, we can assess the system's structural changes and identify phase transitions. This method allows us to detect critical temperatures and phase boundaries.

### Visualization of Phase Behavior
The following plot demonstrates the trend of the **average number of neighbors** across the entire system for different temperatures and over the full simulation duration, along with the variation of kinetic and potential energies over time:
